<a href="https://colab.research.google.com/github/GGSimmons1992/UTYV6k8pXAuLL0yL/blob/main/createEnvironments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as rf
import pickle
from imblearn.over_sampling import SMOTENC
from os.path import exists
from sklearn.preprocessing import StandardScaler
import category_encoders as ce
from sklearn.feature_selection import chi2
from scipy.stats import spearmanr
import json
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/SalesReinforcer/Src/')
import dataPrep
import classicRF

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class BaseSalesEnv(gym.Env):
  def __init__(self, data):
    super().__init__()
    self.data = data
    self.current_step = 0

def reset(self, seed=None, options=None):
    super().reset(seed=seed)
    self.current_step = 0
    return self._get_observation(), {}

def _get_observation(self):
    raise NotImplementedError

def _calculate_reward(self):
    raise NotImplementedError

In [ ]:
class FeatureSelectionEnv(BaseSalesEnv):
  def __init__(self, data):
    super().__init__(data)

    train, dev = train_test_split(data, test_size=0.2, random_state=42)

    self.train = train
    self.dev = dev

    self.n_initial_features = data.shape[1] - 1  # Exclude target column
    self.feature_mask = np.zeros(self.n_initial_features, dtype=np.int8)

    # Load base model
    base_model = classicRF.retrieveModelFromDrive("baseRandomForest.pkl")

    # Hyperparameters for RandomForest
    self.n_estimators = base_model.n_estimators

    # Define options for max_features and set initial index
    self.max_features_options = [None, "sqrt", "log2", 0.25, 0.5, 0.75, 1.0]
    if base_model.max_features is None:
        self.max_features_idx = 0
    elif isinstance(base_model.max_features, str):
        self.max_features_idx = self.max_features_options.index(base_model.max_features)
    else: # assuming it's a float
        # Find the closest option or add it if not present
        if base_model.max_features in self.max_features_options:
            self.max_features_idx = self.max_features_options.index(base_model.max_features)
        else:
            self.max_features_options.append(base_model.max_features)
            self.max_features_options.sort() # Keep it sorted if you want
            self.max_features_idx = self.max_features_options.index(base_model.max_features)

    self.criterion_options = ["gini", "entropy", "log_loss"]
    if base_model.criterion == 'gini':
        self.criterion_idx = 0
    elif base_model.criterion == 'entropy':
        self.criterion_idx = 1
    else:
        self.criterion_idx = 2 # log_loss

    self.max_depth = base_model.max_depth if base_model.max_depth is not None else 10 # Default if None in base model

    # Penalty for adding features
    self.feature_penalty_weight = 0.01 # Adjustable parameter

    # Define actions:
    # 0 to n_initial_features-1: Toggle feature i
    # n_initial_features: Increase n_estimators
    # n_initial_features + 1: Decrease n_estimators
    # n_initial_features + 2: Cycle max_features to next option
    # n_initial_features + 3: Cycle max_features to previous option
    # n_initial_features + 4: Cycle criterion to next option
    # n_initial_features + 5: Cycle criterion to previous option
    # n_initial_features + 6: Increase max_depth
    # n_initial_features + 7: Decrease max_depth
    self.action_space = spaces.Discrete(self.n_initial_features + 8)
    self.observation_space = spaces.Dict({
        "feature_mask": spaces.MultiBinary(self.n_initial_features),
        "n_estimators": spaces.Box(low=1, high=np.inf, shape=(1,), dtype=np.int32),
        "max_features_idx": spaces.Discrete(len(self.max_features_options)),
        "criterion_idx": spaces.Discrete(len(self.criterion_options)),
        "max_depth": spaces.Box(low=1, high=np.inf, shape=(1,), dtype=np.int32)
    })


  def _get_observation(self):
    return {
        "feature_mask": self.feature_mask.copy(),
        "n_estimators": np.array([self.n_estimators], dtype=np.int32),
        "max_features_idx": self.max_features_idx,
        "criterion_idx": self.criterion_idx,
        "max_depth": np.array([self.max_depth], dtype=np.int32)
    }

  def _calculate_reward(self):
    # Select features based on the mask
    selected_feature_indices = np.where(self.feature_mask == 1)[0]
    if len(selected_feature_indices) == 0:
        return 0.0 # No features selected, reward is 0

    # Get feature names from the training data
    feature_columns = self.train.columns[:-1] # Exclude target
    selected_features = feature_columns[selected_feature_indices]

    X_train = self.train[selected_features]
    y_train = self.train.iloc[:, -1]
    X_dev = self.dev[selected_features]
    y_dev = self.dev.iloc[:, -1]

    # Initialize model with current hyperparameters
    current_max_features = self.max_features_options[self.max_features_idx]
    current_criterion = self.criterion_options[self.criterion_idx]

    model = rf(
        n_estimators=self.n_estimators,
        max_features=current_max_features,
        criterion=current_criterion,
        max_depth=self.max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_dev)
    reward = f1_score(y_dev, predictions)

    # Apply penalty for the number of selected features
    reward -= len(selected_feature_indices) * self.feature_penalty_weight

    return reward

  def step(self, action):
    terminated = False
    truncated = False

    if action < self.n_initial_features:
      # Toggle feature
      self.feature_mask[action] = 1 - self.feature_mask[action]
    elif action == self.n_initial_features:
      # Increase n_estimators
      self.n_estimators = min(self.n_estimators + 10, 500) # Cap at 500
    elif action == self.n_initial_features + 1:
      # Decrease n_estimators
      self.n_estimators = max(self.n_estimators - 10, 10) # Min at 10
    elif action == self.n_initial_features + 2:
      # Cycle max_features to next option
      self.max_features_idx = (self.max_features_idx + 1) % len(self.max_features_options)
    elif action == self.n_initial_features + 3:
      # Cycle max_features to previous option
      self.max_features_idx = (self.max_features_idx - 1 + len(self.max_features_options)) % len(self.max_features_options)
    elif action == self.n_initial_features + 4:
      # Cycle criterion to next option
      self.criterion_idx = (self.criterion_idx + 1) % len(self.criterion_options)
    elif action == self.n_initial_features + 5:
      # Cycle criterion to previous option
      self.criterion_idx = (self.criterion_idx - 1 + len(self.criterion_options)) % len(self.criterion_options)
    elif action == self.n_initial_features + 6:
      # Increase max_depth
      self.max_depth = min(self.max_depth + 1, 50) # Cap at 50
    elif action == self.n_initial_features + 7:
      # Decrease max_depth
      self.max_depth = max(self.max_depth - 1, 1) # Min at 1

    reward = self._calculate_reward()

    return (
        self._get_observation(),
        reward,
        terminated,
        truncated,
        {}
    )

In [ ]:
class BusinessActionEnv(BaseSalesEnv):
  def __init__(self, data):
    super().__init__(data)

    self.actions = [
        "call",
        "schedule_demo",
        "request_survey",
        "signup_platform",
        "assign_account_manager"
    ]

    self.action_space = spaces.Discrete(len(self.actions))

    # Depends on how you represent a customer's state
    self.observation_space = ...

  def _get_observation(self):
    return ...

  def _calculate_reward(self):
      return ...

  def step(self, action):
      business_action = self.actions[action]

      # Apply/simulate action
      ...

      reward = self._calculate_reward()

      return (
          self._get_observation(),
          reward,
          False,
          False,
          {}
      )

In [ ]:
def main():
  train = dataPrep.retrieveCSVFromDrive("SalesReinforcerTrain.csv")
  test = dataPrep.retrieveCSVFromDrive("SalesReinforcerTest.csv")

  columns = train.columns
  print(list(columns))

  dataEnviroment = FeatureSelectionEnv(train)
  businessActionEnvironment = BusinessActionEnv(train)




In [ ]:
if __name__ == "__main__":
  main()

['ID', 'First Contact', 'Last Contact', 'First Call', 'Signed up for a demo', 'Filled in customer survey', 'Did sign up to the platform', 'Account Manager assigned', 'First ContactYear', 'First ContactMonth', 'First ContactDay', 'Last ContactYear', 'Last ContactMonth', 'Last ContactDay', 'First CallYear', 'First CallMonth', 'First CallDay', 'Signed up for a demoYear', 'Signed up for a demoMonth', 'Signed up for a demoDay', 'Filled in customer surveyYear', 'Filled in customer surveyMonth', 'Filled in customer surveyDay', 'Did sign up to the platformYear', 'Did sign up to the platformMonth', 'Did sign up to the platformDay', 'Account Manager assignedYear', 'Account Manager assignedMonth', 'Account Manager assignedDay', 'Country_Italy', 'Country_UK', 'Country_USA', 'Country_Canada', 'Country_Sweden', 'Country_0', 'Country_Saudi Arabia', 'Country_France', 'Country_Morocco', 'Country_Australia', 'Country_Singapore', 'Country_South Africa', 'Country_Ireland', 'Country_Spain', 'Country_German